# LangGraph Company Research and Outreach Workflow

## 1. Project Overview

**Task:** Build a **multi-node LangGraph pipeline** that researches a target company, summarizes the lead, drafts a personalized outreach message, and runs a QA review before sending — all modeled as an explicit state graph.

**Why this matters:** Sales, partnerships, and business development all follow the same sequence: research → synthesize → draft → review. Turning this into a state graph makes every step inspectable, testable, and revisable — unlike a single mega-prompt that hides intermediate reasoning.

**Pipeline Nodes:**
1. **Research Company** — gather profile data, recent news, and product info
2. **Summarize Lead** — distill the research into a concise lead brief
3. **Draft Outreach** — write a personalized outreach email using the brief
4. **QA Review** — check the draft for factual accuracy, tone, and completeness
5. **Conditional Routing** — approve, revise, or reject based on QA score

**Stack:**
- `LangGraph` — state graph orchestration
- `ChatOllama` + `qwen3.5:9b` — LLM (local, no API keys)

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Design a **TypedDict state schema** that carries data through the entire graph |
| 2 | Build **single-responsibility nodes** that read specific fields and write specific outputs |
| 3 | Understand **state passing** — how partial updates merge into accumulating state |
| 4 | Implement **conditional edges** that branch on QA results |
| 5 | Add a **revision loop** so the drafter can improve based on QA feedback |
| 6 | Inspect **intermediate state** at each graph step |
| 7 | Evaluate the full pipeline on multiple leads |

## 3. Node Design Principles

Good LangGraph nodes follow three rules:

### Rule 1: Single Responsibility
Each node does **one thing** and names its output clearly.

```
  BAD:  one node that researches, summarizes, and drafts
  GOOD: three separate nodes — research_company, summarize_lead, draft_outreach
```

### Rule 2: Read Explicitly, Write Partially
A node reads specific fields from state and returns **only the fields it changed**. LangGraph merges the partial update into the full state automatically.

```python
  def summarize_lead(state: PipelineState) -> dict:
      # READ from state
      research = state["company_research"]
      # DO the work
      summary = llm_summarize(research)
      # WRITE only what changed
      return {"lead_summary": summary, "current_node": "summarize_lead"}
```

### Rule 3: Stateless Execution
Nodes should not keep internal mutable state. Everything lives in the graph state so it can be inspected, serialized, and resumed.

## 4. State Passing — How It Works

```
  ┌─────────────────────────────────────────────────────────────────┐
  │                    STATE PASSING IN LANGGRAPH                   │
  ├─────────────────────────────────────────────────────────────────┤
  │                                                                 │
  │  STEP 1: Node A runs                                           │
  │    state_before = {company: "Acme", research: "", summary: ""}  │
  │    node_output   = {research: "Acme is a SaaS company..."}      │
  │    state_after   = {company: "Acme", research: "Acme is...",   │
  │                     summary: ""}  ← merged automatically       │
  │                                                                 │
  │  STEP 2: Node B runs                                           │
  │    state_before = (full state from step 1)                      │
  │    node_output   = {summary: "SaaS platform, 200 employees"}   │
  │    state_after   = {company: "Acme", research: "Acme is...",   │
  │                     summary: "SaaS platform, 200 employees"}   │
  │                                                                 │
  │  KEY INSIGHT: Each node only returns what it changed.           │
  │  LangGraph handles the merge. This prevents accidental         │
  │  overwrites and keeps node logic clean.                         │
  └─────────────────────────────────────────────────────────────────┘
```

## 5. Our Pipeline Graph

```
  START
    │
    ▼
  ┌──────────────────┐
  │ research_company  │  gather profile, news, and product info
  └────────┬─────────┘
           │
           ▼
  ┌──────────────────┐
  │ summarize_lead    │  distill research into a lead brief
  └────────┬─────────┘
           │
           ▼
  ┌──────────────────┐
  │ draft_outreach    │  write personalized email
  └────────┬─────────┘
           │
           ▼
  ┌──────────────────┐
  │ qa_review         │  check facts, tone, completeness
  └────────┬─────────┘
           │
      conditional
      ┌────┴────┐
      ▼         ▼
   approve    revise ──→ back to draft_outreach
      │
      ▼
  ┌──────────────────┐
  │ finalize_output   │  format the approved outreach package
  └────────┬─────────┘
           │
           ▼
         END
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langgraph

print("Dependencies: langchain, langgraph")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports & Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from typing import TypedDict

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

LLM_MODEL = "qwen3.5:9b"

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"LangGraph: StateGraph, START, END imported")

## 8. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.2) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 100):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM check: {test[:30]}")

---

# Part A — Data & State Schema

## 9. Target Companies (Simulated CRM Data)

In production these would come from a CRM or enrichment API. Here we use hand-crafted company profiles so the notebook is fully self-contained and reproducible.

In [ ]:
TARGET_COMPANIES = [
    {
        "company_name": "NovaTech Solutions",
        "industry": "B2B SaaS — Developer Tools",
        "hq": "Austin, TX",
        "employees": 220,
        "funding": "Series B — $38M",
        "products": "CI/CD platform for mobile teams, automated testing suite, deployment analytics dashboard",
        "recent_news": "Announced SOC 2 Type II certification. Launched EU region. Hired new VP Engineering from Stripe.",
        "pain_points": "Scaling enterprise sales; compliance requirements growing; need to shorten sales cycle",
        "contact_name": "Sarah Chen",
        "contact_title": "Head of Partnerships",
    },
    {
        "company_name": "GreenLeaf Analytics",
        "industry": "Climate Tech — Carbon Accounting",
        "hq": "Berlin, Germany",
        "employees": 85,
        "funding": "Series A — $12M",
        "products": "Carbon footprint calculator for supply chains, ESG reporting tool, emissions API",
        "recent_news": "Won EU Green Tech Award. Published carbon methodology whitepaper. Partnered with SAP.",
        "pain_points": "Regulatory deadlines for CSRD compliance; small sales team; need integrations with ERP systems",
        "contact_name": "Marco Weber",
        "contact_title": "CEO & Co-Founder",
    },
    {
        "company_name": "MediSync Health",
        "industry": "HealthTech — Clinical Data",
        "hq": "Toronto, Canada",
        "employees": 150,
        "funding": "Series B — $25M",
        "products": "EHR integration middleware, clinical trial data pipeline, patient consent management",
        "recent_news": "HIPAA and PIPEDA certified. Launched real-time data sync for three hospital networks. New CTO from Epic.",
        "pain_points": "Complex procurement cycles in healthcare; interoperability challenges; need faster onboarding",
        "contact_name": "Dr. Priya Sharma",
        "contact_title": "VP Business Development",
    },
]

print(f"Target companies: {len(TARGET_COMPANIES)}")
for c in TARGET_COMPANIES:
    print(f"  {c['company_name']} — {c['industry']} — {c['employees']} employees")

## 10. State Schema

The state is a `TypedDict` that carries data through every node. Each field is written by a specific node and read by downstream nodes.

**State field ownership:**

| Field | Written By | Read By |
|-------|-----------|--------|
| `company_data` | Input | research_company |
| `company_research` | research_company | summarize_lead |
| `lead_summary` | summarize_lead | draft_outreach |
| `outreach_draft` | draft_outreach | qa_review |
| `qa_result` | qa_review | routing function |
| `qa_feedback` | qa_review | draft_outreach (on revision) |
| `revision_count` | qa_review | routing function (safety limit) |
| `final_output` | finalize_output | caller |
| `current_node` | every node | debugging |

In [ ]:
class OutreachState(TypedDict):
    """State schema for the company research and outreach pipeline.

    Design notes:
    - company_data is the raw input from the CRM
    - company_research is the expanded research (LLM-generated)
    - lead_summary is a concise brief derived from the research
    - outreach_draft is the email draft
    - qa_result is 'approve' or 'revise'
    - qa_feedback contains improvement suggestions if revise
    - revision_count prevents infinite loops
    - final_output is the approved package (only written on approve)
    - current_node is for debugging
    """
    company_data: dict
    company_research: str
    lead_summary: str
    outreach_draft: str
    qa_result: str
    qa_feedback: str
    revision_count: int
    final_output: str
    current_node: str


print("State schema defined: OutreachState")
print("Fields:")
for name, typ in OutreachState.__annotations__.items():
    print(f"  {name}: {typ}")

---

# Part B — Build the Graph Nodes

## 11. Node 1: Research Company

This node takes the raw company data and produces an enriched research brief. In production this might call enrichment APIs (Clearbit, LinkedIn, Crunchbase). Here the LLM synthesizes the provided profile data into a structured research output.

In [ ]:
RESEARCH_SYSTEM = """You are a business research analyst. Given company profile data,
produce a thorough research brief covering:
- Company positioning and competitive landscape
- Growth signals and recent milestones
- Likely priorities and pain points
- Opportunities for partnership or sales engagement
Write in concise, factual paragraphs. Cite specific data points from the profile."""

RESEARCH_PROMPT = """COMPANY PROFILE:
Name: {company_name}
Industry: {industry}
HQ: {hq}
Employees: {employees}
Funding: {funding}
Products: {products}
Recent News: {recent_news}
Known Pain Points: {pain_points}
Contact: {contact_name}, {contact_title}

Write a research brief (4-6 paragraphs) covering positioning, growth signals, priorities, and engagement opportunities."""


def research_company(state: OutreachState) -> dict:
    """Node: Expand raw company data into a research brief."""
    print("  [NODE] research_company")
    data = state["company_data"]
    research = ask(
        RESEARCH_PROMPT.format(**data),
        system=RESEARCH_SYSTEM,
        temperature=0.3,
    )
    print(f"    Research generated: {len(research)} chars")
    return {"company_research": research, "current_node": "research_company"}


print("Node defined: research_company")
print("  READS:  company_data")
print("  WRITES: company_research")

## 12. Node 2: Summarize Lead

Distill the research into a short **lead brief** — the essential points someone needs to know before writing an outreach email.

In [ ]:
SUMMARY_SYSTEM = """Summarize a company research brief into a lead card with these sections:
1. ONE-LINE PITCH: what the company does in one sentence
2. KEY SIGNALS: 3-4 bullet points of growth signals or recent wins
3. PAIN POINTS: 2-3 bullet points of likely pain points
4. ENGAGEMENT ANGLE: 1-2 sentences on the best angle for outreach
Keep it to 200 words maximum."""


def summarize_lead(state: OutreachState) -> dict:
    """Node: Create a concise lead brief from the research."""
    print("  [NODE] summarize_lead")
    summary = ask(
        f"RESEARCH BRIEF:\n{state['company_research'][:2000]}\n\nSummarize into a lead card.",
        system=SUMMARY_SYSTEM,
        temperature=0.2,
    )
    print(f"    Lead summary: {len(summary)} chars")
    return {"lead_summary": summary, "current_node": "summarize_lead"}


print("Node defined: summarize_lead")
print("  READS:  company_research")
print("  WRITES: lead_summary")

## 13. Node 3: Draft Outreach

Write a personalized outreach email using the lead summary, the contact details, and (if available) QA feedback from a previous revision.

In [ ]:
DRAFT_SYSTEM = """Write a professional outreach email. The email must:
- Open with a specific, personalized hook referencing the company's recent news or achievements
- Connect your value proposition to their known pain points
- Include a clear, low-commitment call to action
- Be concise: 150-200 words maximum
- Use professional but warm tone — not salesy or pushy
Do NOT use placeholder brackets like [X] or [YOUR COMPANY]. Write complete sentences."""

DRAFT_PROMPT = """LEAD SUMMARY:
{lead_summary}

CONTACT: {contact_name}, {contact_title} at {company_name}

{revision_section}

Write the outreach email. Start with "Subject:" then the email body."""


def draft_outreach(state: OutreachState) -> dict:
    """Node: Draft a personalized outreach email."""
    print("  [NODE] draft_outreach")
    data = state["company_data"]

    revision_section = ""
    if state.get("qa_feedback") and state.get("revision_count", 0) > 0:
        revision_section = (
            f"REVISION FEEDBACK (incorporate these improvements):\n"
            f"{state['qa_feedback']}\n\n"
            f"PREVIOUS DRAFT (improve, do not start from scratch):\n"
            f"{state['outreach_draft'][:500]}"
        )
        print(f"    Revision round {state['revision_count']} — incorporating QA feedback")

    draft = ask(
        DRAFT_PROMPT.format(
            lead_summary=state["lead_summary"][:1000],
            contact_name=data["contact_name"],
            contact_title=data["contact_title"],
            company_name=data["company_name"],
            revision_section=revision_section,
        ),
        system=DRAFT_SYSTEM,
        temperature=0.4,
    )
    print(f"    Draft generated: {len(draft)} chars")
    return {"outreach_draft": draft, "current_node": "draft_outreach"}


print("Node defined: draft_outreach")
print("  READS:  lead_summary, company_data, qa_feedback (if revision)")
print("  WRITES: outreach_draft")

## 14. Node 4: QA Review

The QA reviewer checks the draft against the research and scores it. If the score is below threshold or issues are found, it triggers a revision.

**QA dimensions:**
- **Factual accuracy** — does the email match the research?
- **Personalization** — does it reference specific company details?
- **Tone** — professional without being too formal or too casual?
- **Call to action** — clear and low-commitment?
- **Length** — within target range?

In [ ]:
QA_SYSTEM = """You are a QA reviewer for sales outreach emails.
Score the email on five dimensions (1-5 each) and decide: approve or revise.
Return valid JSON only."""

QA_PROMPT = """LEAD SUMMARY (ground truth):
{lead_summary}

OUTREACH DRAFT:
{draft}

Score each dimension 1-5 and provide an overall decision.
Return JSON:
{{
  "factual_accuracy": 1-5,
  "personalization": 1-5,
  "tone": 1-5,
  "call_to_action": 1-5,
  "length_appropriate": 1-5,
  "average_score": float,
  "decision": "approve" or "revise",
  "feedback": "specific improvements needed (empty string if approved)"
}}"""


def qa_review(state: OutreachState) -> dict:
    """Node: QA-review the outreach draft against the lead summary."""
    print("  [NODE] qa_review")

    raw = ask(
        QA_PROMPT.format(
            lead_summary=state["lead_summary"][:1000],
            draft=state["outreach_draft"][:1500],
        ),
        system=QA_SYSTEM,
        temperature=0.1,
    )
    result = parse_json(raw)

    if not result:
        result = {
            "factual_accuracy": 3, "personalization": 3, "tone": 3,
            "call_to_action": 3, "length_appropriate": 3,
            "average_score": 3.0, "decision": "approve",
            "feedback": "JSON parse failed — defaulting to approve",
        }

    # Compute average if not provided
    dims = ["factual_accuracy", "personalization", "tone", "call_to_action", "length_appropriate"]
    scores = [result.get(d, 3) for d in dims]
    avg = sum(scores) / len(scores)
    result["average_score"] = round(avg, 2)

    # Override decision based on score threshold
    if avg < 3.5:
        result["decision"] = "revise"
    elif avg >= 4.0:
        result["decision"] = "approve"

    revision_count = state.get("revision_count", 0)

    print(f"    Scores: {', '.join(f'{d}={result.get(d)}' for d in dims)}")
    print(f"    Average: {result['average_score']} → {result['decision'].upper()}")

    return {
        "qa_result": result["decision"],
        "qa_feedback": result.get("feedback", ""),
        "revision_count": revision_count + 1,
        "current_node": "qa_review",
    }


print("Node defined: qa_review")
print("  READS:  lead_summary, outreach_draft")
print("  WRITES: qa_result, qa_feedback, revision_count")

## 15. Node 5: Finalize Output

After approval, compile the entire outreach package into a clean final output.

In [ ]:
def finalize_output(state: OutreachState) -> dict:
    """Node: Compile the approved outreach package."""
    print("  [NODE] finalize_output")
    data = state["company_data"]

    final = (
        f"=== OUTREACH PACKAGE ==="
        f"\n\nCompany: {data['company_name']}"
        f"\nContact: {data['contact_name']}, {data['contact_title']}"
        f"\nIndustry: {data['industry']}"
        f"\nRevision Rounds: {state.get('revision_count', 0)}"
        f"\n\n--- LEAD BRIEF ---"
        f"\n{state['lead_summary']}"
        f"\n\n--- OUTREACH EMAIL ---"
        f"\n{state['outreach_draft']}"
    )
    print(f"    Final package: {len(final)} chars")
    return {"final_output": final, "current_node": "finalize_output"}


print("Node defined: finalize_output")
print("  READS:  company_data, lead_summary, outreach_draft, revision_count")
print("  WRITES: final_output")

---

# Part C — Conditional Routing & Graph Assembly

## 16. Routing After QA Review

The conditional edge inspects `qa_result` and `revision_count`:
- If `approve` → go to `finalize_output`
- If `revise` and under the revision limit → loop back to `draft_outreach`
- If `revise` but at revision limit → force approval to prevent infinite loops

In [ ]:
MAX_REVISIONS = 3


def route_after_qa(state: OutreachState) -> str:
    """Conditional edge: route based on QA result and revision count."""
    decision = state.get("qa_result", "approve")
    revisions = state.get("revision_count", 0)

    if decision == "approve":
        print(f"    [ROUTE] approved → finalize_output")
        return "finalize_output"

    if revisions >= MAX_REVISIONS:
        print(f"    [ROUTE] max revisions ({MAX_REVISIONS}) reached → forcing finalize_output")
        return "finalize_output"

    print(f"    [ROUTE] revise → draft_outreach (revision {revisions})")
    return "draft_outreach"


print("Routing function defined: route_after_qa")
print(f"  Max revisions: {MAX_REVISIONS}")
print("  approve            → finalize_output")
print("  revise (under cap) → draft_outreach")
print("  revise (at cap)    → finalize_output")

## 17. Assemble the StateGraph

Now we wire nodes and edges together. Pay attention to how each `add_edge` and `add_conditional_edges` call matches the architecture diagram above.

In [ ]:
# Build the graph
workflow = StateGraph(OutreachState)

# Add all five nodes
workflow.add_node("research_company", research_company)
workflow.add_node("summarize_lead", summarize_lead)
workflow.add_node("draft_outreach", draft_outreach)
workflow.add_node("qa_review", qa_review)
workflow.add_node("finalize_output", finalize_output)

# Linear edges: START → research → summarize → draft → qa
workflow.add_edge(START, "research_company")
workflow.add_edge("research_company", "summarize_lead")
workflow.add_edge("summarize_lead", "draft_outreach")
workflow.add_edge("draft_outreach", "qa_review")

# Conditional edge after QA — the branching point
workflow.add_conditional_edges(
    "qa_review",
    route_after_qa,
    {
        "finalize_output": "finalize_output",
        "draft_outreach": "draft_outreach",
    },
)

# Final edge
workflow.add_edge("finalize_output", END)

# Compile into a runnable
app = workflow.compile()

print("Graph compiled!")
print(f"Nodes: {list(workflow.nodes.keys())}")
print("\nFlow:")
print("  START → research_company → summarize_lead → draft_outreach → qa_review")
print("  qa_review ──(approve)──→ finalize_output → END")
print("  qa_review ──(revise)───→ draft_outreach (loop)")

In [ ]:
# Visualize graph structure
try:
    mermaid_str = app.get_graph().draw_mermaid()
    print("Mermaid diagram:")
    print(mermaid_str)
except Exception as e:
    print(f"Mermaid rendering not available: {e}")
    print("\nGraph (text):")
    print("  __start__ --> research_company --> summarize_lead")
    print("  summarize_lead --> draft_outreach --> qa_review")
    print("  qa_review --(approve)--> finalize_output --> __end__")
    print("  qa_review --(revise)--> draft_outreach")

---

# Part D — Execute the Pipeline

## 18. Run on First Company

In [ ]:
def make_initial_state(company: dict) -> OutreachState:
    return {
        "company_data": company,
        "company_research": "",
        "lead_summary": "",
        "outreach_draft": "",
        "qa_result": "",
        "qa_feedback": "",
        "revision_count": 0,
        "final_output": "",
        "current_node": "start",
    }


print(f"Running pipeline for: {TARGET_COMPANIES[0]['company_name']}")
print("=" * 70)

result_1 = app.invoke(
    make_initial_state(TARGET_COMPANIES[0]),
    {"recursion_limit": 20},
)

print(f"\nPipeline complete.")
print(f"  QA result: {result_1['qa_result']}")
print(f"  Revision rounds: {result_1['revision_count']}")
print(f"  Final output: {len(result_1['final_output'])} chars")

## 19. Inspect Intermediate State

One of the biggest advantages of state graphs: you can inspect what each node produced.

In [ ]:
print("STATE INSPECTION — Company 1")
print("=" * 80)

print("\n[1] COMPANY RESEARCH (first 300 chars):")
wrap_print(result_1["company_research"][:300])

print("\n[2] LEAD SUMMARY:")
wrap_print(result_1["lead_summary"][:500])

print("\n[3] OUTREACH DRAFT:")
wrap_print(result_1["outreach_draft"][:600])

print(f"\n[4] QA RESULT: {result_1['qa_result']}")
if result_1["qa_feedback"]:
    print(f"    Feedback: {result_1['qa_feedback'][:150]}")

## 20. Run on All Companies

In [ ]:
print("RUNNING ON ALL TARGET COMPANIES")
print("=" * 70)

all_results = []

for i, company in enumerate(TARGET_COMPANIES, 1):
    print(f"\n{'=' * 70}")
    print(f"COMPANY {i}/{len(TARGET_COMPANIES)}: {company['company_name']}")
    print("=" * 70)

    result = app.invoke(
        make_initial_state(company),
        {"recursion_limit": 20},
    )
    all_results.append(result)

    print(f"  → QA: {result['qa_result']} | Revisions: {result['revision_count']} | "
          f"Output: {len(result['final_output'])} chars")

print(f"\nAll {len(TARGET_COMPANIES)} companies processed.")

---

# Part E — View Final Outreach Packages

## 21. Final Output Display

In [ ]:
for i, result in enumerate(all_results, 1):
    company = TARGET_COMPANIES[i - 1]
    print(f"\n{'#' * 80}")
    print(f"# COMPANY {i}: {company['company_name']}")
    print(f"{'#' * 80}")
    wrap_print(result["final_output"])
    print()

---

# Part F — Pipeline Evaluation

## 22. Cross-Company Metrics

In [ ]:
print("PIPELINE METRICS")
print("=" * 80)
print(f"{'Company':<25} {'QA Result':<12} {'Revisions':<12} {'Output Len':<12}")
print("-" * 61)

for r, c in zip(all_results, TARGET_COMPANIES):
    print(f"{c['company_name']:<25} {r['qa_result']:<12} {r['revision_count']:<12} {len(r['final_output']):<12}")

# Aggregate
avg_revisions = sum(r["revision_count"] for r in all_results) / len(all_results)
approved = sum(1 for r in all_results if r["qa_result"] == "approve")
print(f"\n  Approval rate:      {approved}/{len(all_results)}")
print(f"  Average revisions:  {avg_revisions:.1f}")

## 23. Node Design Recap

Let's trace exactly what each node reads and writes, showing the state-passing pattern in action.

```
  Node                READS                      WRITES
  ─────────────────── ────────────────────────── ──────────────────────
  research_company    company_data               company_research
  summarize_lead      company_research           lead_summary
  draft_outreach      lead_summary, company_data outreach_draft
                      (+ qa_feedback on revision)
  qa_review           lead_summary, outreach_draft qa_result, qa_feedback,
                                                   revision_count
  finalize_output     company_data, lead_summary, final_output
                      outreach_draft, revision_count

  RULE: Each node returns ONLY its WRITES.
  LangGraph merges the partial dict into the full state.
  Downstream nodes see the accumulated state from all prior nodes.
```

In [ ]:
# Print state field sizes for Company 1 to show accumulation
r = all_results[0]
print("STATE ACCUMULATION (Company 1)")
print("=" * 60)
fields = [
    ("company_data", json.dumps(r["company_data"])),
    ("company_research", r["company_research"]),
    ("lead_summary", r["lead_summary"]),
    ("outreach_draft", r["outreach_draft"]),
    ("qa_result", r["qa_result"]),
    ("qa_feedback", r["qa_feedback"]),
    ("final_output", r["final_output"]),
]

cumulative = 0
for name, content in fields:
    size = len(content)
    cumulative += size
    bar = "|" * min(size // 50, 40)
    print(f"  {name:<22} {size:>5} chars  {bar}")

print(f"\n  Total state: {cumulative:,} chars")
print(f"  This grows as each node adds its output.")

## 24. Step-by-Step Streaming View

Using `app.stream()` lets us observe each node as it completes — useful for UIs and debugging.

In [ ]:
print("STREAMING EXECUTION — Company 2")
print("=" * 60)

step = 0
for chunk in app.stream(make_initial_state(TARGET_COMPANIES[1]), {"recursion_limit": 20}):
    step += 1
    for node_name, node_output in chunk.items():
        keys_written = [k for k in node_output.keys() if k != "current_node"]
        sizes = {k: len(str(node_output[k])) for k in keys_written}
        print(f"  Step {step}: {node_name} → wrote {keys_written} (sizes: {sizes})")

print(f"\nTotal steps: {step}")

---

## 25. Common Mistakes in Node Design

| Mistake | Problem | Fix |
|---------|---------|-----|
| Returning the full state from a node | Overwrites fields written by other nodes | Return only changed fields |
| Node does too many things | Hard to debug, impossible to reorder | Split into single-responsibility nodes |
| No revision cap on loops | Infinite revisions if QA is always unsatisfied | Add `MAX_REVISIONS` guard |
| QA scores are vague | "Looks good" doesn't help the drafter improve | Use specific dimensions and numeric scores |
| State fields are ambiguous | Downstream nodes misinterpret data | Document read/write ownership per node |

## 26. Production Extensions

1. **Enrichment APIs** — replace simulated profiles with Clearbit, LinkedIn Sales Navigator, or Apollo
2. **Email sending** — add a `send_email` node after finalize, gated by human approval
3. **CRM integration** — write results back to HubSpot or Salesforce with activity logging
4. **A/B testing** — generate two draft variants and let the QA reviewer pick the stronger one
5. **Batch processing** — stream all companies through the graph in parallel with checkpointing

## 27. Exercises

### Exercise 1
Add a **tone selector**: let the user specify "formal", "casual", or "technical" in the initial state, and modify the draft node to respect that tone. Verify the QA node can detect tone mismatches.

### Exercise 2
Implement **multi-channel output**: extend `finalize_output` to produce both an email draft and a LinkedIn connection request message. Both should reference the same research but adapt to the channel.

### Exercise 3
Add a **competing draft node**: create two parallel drafts with different temperatures, then use the QA reviewer to pick the better one. You'll need to modify the state schema to hold two drafts.

### Mini Challenge
Build a **follow-up workflow**: after the initial outreach is approved, add nodes that check for a reply (simulated), and if no reply after N days, generate a follow-up email that references the original message without being repetitive.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Companies processed:  {len(all_results)}")
print(f"Approval rate:        {approved}/{len(all_results)}")
print(f"Avg revisions:        {avg_revisions:.1f}")
print()
print("Graph topology:")
print("  START → research_company → summarize_lead → draft_outreach")
print("  → qa_review ──(approve)──→ finalize_output → END")
print("  → qa_review ──(revise)───→ draft_outreach (loop)")
print()
print("Components built:")
print("  1. research_company  — enriches raw CRM data into research brief")
print("  2. summarize_lead    — distills research into a concise lead card")
print("  3. draft_outreach    — writes personalized email from the lead brief")
print("  4. qa_review         — scores on 5 dimensions, routes approve/revise")
print("  5. finalize_output   — compiles approved outreach package")
print("  6. route_after_qa    — conditional edge with revision cap")
print()
print("Key patterns demonstrated:")
print("  • TypedDict state schema with field ownership")
print("  • Partial state updates (nodes return only changed fields)")
print("  • Conditional routing based on QA scores")
print("  • Revision loop with safety cap")
print("  • Step-by-step streaming for debugging")
print("  • State inspection at every intermediate step")

## 28. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Each node should have one job** — research, summarize, draft, or review, never all at once |
| 2 | **Return only what changed** — partial state updates prevent accidental field overwrites |
| 3 | **State accumulates through the graph** — downstream nodes see all prior outputs without re-computing |
| 4 | **Conditional edges encode decision logic** — QA result routes to either finalize or revision |
| 5 | **Document field ownership** — which node writes each field, which nodes read it |
| 6 | **Cap revision loops** — infinite loops are easy to create without a `MAX_REVISIONS` guard |
| 7 | **Streaming shows progress** — `app.stream()` lets you observe each node as it completes |
| 8 | **QA should score dimensions, not just approve/reject** — specific feedback drives better revisions |
| 9 | **State inspection is the debugging story** — print intermediate fields to understand failures |
| 10 | **The graph is the contract** — it makes the pipeline explicit, testable, and auditable |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*